# Исследование вращающихся лучей

Ноутбук — клиент пакета. Меняйте `config`, а алгоритмы сохраняйте в `src/diffusion_lab`.
Короткое обучение ниже проверяет эксперимент; качество оценивайте метриками и более долгим запуском.


In [ ]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from diffusion_lab.experiment import load_config, build_experiment
from diffusion_lab.visualization import (Grid, evaluate_field, plot_dataset,
    plot_vector_field, recover_potential, plot_potential, interactive_field)
from diffusion_lab.metrics import evaluate_samples

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
config = load_config(ROOT / "configs/rays.yaml")
config["environment"]["n_samples"] = 1024
config["model"].update(hidden_dim=64, num_layers=3)
config["scheduler"]["num_train_timesteps"] = 100
config["training"].update(epochs=8, batch_size=128, ema_decay=0.9)
dataset, pipeline, trainer = build_experiment(config)
print(dataset.sample_shape, dataset.condition_kind, dataset.num_classes)


## Данные и обучение

Для других размерностей замените `environment.name` на `gaussian_mixture_1d` или `helix_3d`.
`conditioner` можно переключить между `one_hot`, `embedding` и собственным encoder.
Для flow нужны одновременно `scheduler.kind="flow"` и `training.objective="rectified_flow"`.


In [ ]:
plot_dataset(dataset.samples, dataset.labels, title="Rotating rays: training distribution")
plt.show()
trainer.fit(dataset)
print(trainer.history[-1])


In [ ]:
conditions = torch.arange(256) % dataset.num_classes
with trainer.ema_weights():
    generated = pipeline(batch_size=256, conditions=conditions,
        num_inference_steps=30, guidance_scale=1.5,
        generator=torch.Generator().manual_seed(42), record_history=True)
print(evaluate_samples(dataset, generated.samples, conditions))
plot_dataset(generated.samples, conditions, title="Generated rays (short tutorial training)")
plt.show()


## Поле и потенциал

Query-точки уже считаются x_t: повторное зашумление не применяется.
`score`, `epsilon`, `denoising` имеют разные значения и масштабы.
Из NN-поля можно получить лишь приближённый потенциал, если residual/curl ненулевые.
В 3D строится объёмная сетка и показывается срез; у модели путей пространственное
поле не определено этим API — смотрите ноутбук навигации.


In [ ]:
reference = dataset.samples.numpy()
bounds = [(float(reference[:, d].min()-0.25), float(reference[:, d].max()+0.25))
          for d in range(reference.shape[-1])]
grid = Grid.from_bounds(bounds, resolution=32 if len(bounds) < 3 else 12)
fields = {}
with trainer.ema_weights():
    for timestep in (10, 35, 65):
        fields[f"condition=0; t={timestep}"] = evaluate_field(
            lambda points: pipeline.vector_field(points, timestep, conditions=0,
                kind="score", guidance_scale=1.5), grid)
field = fields["condition=0; t=35"]
plot_vector_field(grid, field, samples=reference, stride=2, streamlines=grid.ndim == 2)
plt.show()
potential = recover_potential(grid, field)
print("residual:", potential.relative_residual, "curl RMS:", potential.curl_rms)
print(potential.message)
plot_potential(grid, potential)
plt.show()


In [ ]:
interactive = interactive_field(grid, fields, samples=reference[:500], stride=2)
interactive


## Сохранение и дальнейшие опыты

Для полного воспроизводимого запуска используйте CLI: он сохранит pipeline,
optimizer/EMA/RNG checkpoint, конфиг, manifest, метрики и SVG/PNG.
```bash
uv run diffusion-lab train --config configs/rays.yaml --output outputs/experiment
uv run diffusion-lab sweep --config configs/rays.yaml --parameter model.conditioner \
  --values one_hot embedding --epochs 30 --output outputs/compare
```
